# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Import necessary modules and set the working directory to the repository root.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython


from IPython.display import display
import os 

# replace this with the path to the directory on your machine
os.chdir('../..')
import bacpipe

---
## 2. Using a Custom Model
Define your own model by subclassing `ModelBaseClass` and plug it directly into bacpipe's pipeline.

In [ ]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000
    SEGMENT_LENGTH = 48000

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio * 10

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # return array needs to be 2D!
        mel_spec = mel_spec.reshape(
            [len(mel_spec), mel_spec.shape[-2] * mel_spec.shape[-1]]
            )
        return mel_spec

---
## 3. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [ ]:
from bacpipe.model_pipelines.feature_extractors.birdnet import Model

class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input ** 2
        return self.embeds(input, training=False)

---
## 4. Probing Pipeline
Train a linear probe on top of frozen embeddings to evaluate how well the representations capture your specific ground truth annotations.

In [ ]:
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

embeds = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet2',
    use_folder_structure=True,
    CustomModel=MyBirdNETModel
).embeddings(return_type='array')

probe, label2idx, metrics = bacpipe.probing_pipeline('birdnet', gt, embeds)

display(type(probe))
display(label2idx)
display(metrics)

---
## 5. Probe Inference
Load a previously trained linear probe and run inference to generate new predictions from your embeddings.

In [ ]:
probe, label2idx = bacpipe.prepare_probe_inference('birdnet')

predictions = bacpipe.run_probe_inference(
    'birdnet', 
    probe, 
    0.5, 
    embeds, 
    device='cuda', 
    return_binary_presence=False
)

---
## 6. Clustering Pipeline
Run bacpipe's built-in clustering evaluation pipeline to group embeddings and compare the clusters against your ground truth labels.

In [ ]:
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

embeds = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet',
    use_folder_structure=True
).embeddings(return_type='array')

clusterings, metrics = bacpipe.clustering_pipeline('birdnet', gt, embeds)

---
## 7. Custom Clustering & Evaluation
Inject your own custom clustering algorithms (like sklearn's KMeans) and evaluate their performance using metrics like the Silhouette score.

In [ ]:
from sklearn.cluster import KMeans

# Make sure embeddings are generated
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

# Run custom clustering without ground truth
clustered_points = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}
)

# Run custom clustering with ground truth for alignment/evaluation
clustered_points_gt = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}, 
    ground_truth=gt['label:species']
)

# Evaluate embedding separation using Silhouette score
metrics = bacpipe.eval_with_silhouette(
    embeds, 
    ground_truth=gt['label:species']
)